# 01 - Tao embedding bge-m3 cho chunks.jsonl

Input: dataset `vn-labor-law-chunks` (chunks.jsonl).
Output: `/kaggle/working/embeddings.parquet` voi cot `id`, `dense` (list[float] 1024 chieu),
`sparse_indices` (list[int]), `sparse_values` (list[float]) -- dung dinh dang tach roi thay vi
dict de tuong thich tot voi parquet va khop truc tiep voi sparse vector cua Qdrant.

In [ ]:
!pip install -q FlagEmbedding==1.4.2

In [ ]:
import json

INPUT_PATH = "/kaggle/input/vn-labor-law-chunks/chunks.jsonl"

records = [json.loads(line) for line in open(INPUT_PATH, encoding="utf-8")]
ids = [r["id"] for r in records]
texts = [r["noi_dung"] for r in records]
print(f"So chunk: {len(texts)}")

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
output = model.encode(
    texts,
    batch_size=16,
    max_length=1024,
    return_dense=True,
    return_sparse=True,
)

In [ ]:
import pandas as pd

dense = output["dense_vecs"]
sparse = output["lexical_weights"]  # list[dict[str(token_id) -> weight]]

sparse_indices = [[int(k) for k in d.keys()] for d in sparse]
sparse_values = [[float(v) for v in d.values()] for d in sparse]

df = pd.DataFrame({
    "id": ids,
    "dense": [row.tolist() for row in dense],
    "sparse_indices": sparse_indices,
    "sparse_values": sparse_values,
})
df.to_parquet("/kaggle/working/embeddings.parquet", index=False)
print(df.shape)
df.head()